# Phase 4: Simple Research Paper Preprocessing
This notebook follows the Lab1 style: lowercase, tokenize, remove punctuation, remove stopwords, optionally lemmatize, and return clean tokens.

In [15]:
import importlib
import sys
from pathlib import Path

import nltk
import pandas as pd

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing as preprocessing
preprocessing = importlib.reload(preprocessing)

from src.preprocessing import (
    OUTPUT_COLUMNS,
    REQUIRED_COLUMNS,
    build_preprocessed_dataset,
    clean_text,
    preprocess_dataframe,
    preprocess_document,
    remove_stopwords,
    tokenize_text,
    validate_required_columns,
)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [11]:
# Raw text example
raw_text = "BERT, CNN, HPC, GPU, NLP and LLM models are useful!!!"

print("raw text =", raw_text)
print("lowercase =", raw_text.lower())
print("tokens =", tokenize_text(raw_text))
print("without stopwords =", remove_stopwords(tokenize_text(raw_text)))
print("clean text =", clean_text(raw_text))
print("lemmatized text =", clean_text(raw_text, lemmatize=True))


raw text = BERT, CNN, HPC, GPU, NLP and LLM models are useful!!!
lowercase = bert, cnn, hpc, gpu, nlp and llm models are useful!!!
tokens = ['bert', 'cnn', 'hpc', 'gpu', 'nlp', 'and', 'llm', 'models', 'are', 'useful']
without stopwords = ['bert', 'cnn', 'hpc', 'gpu', 'nlp', 'llm', 'models', 'useful']
clean text = bert cnn hpc gpu nlp llm models useful
lemmatized text = bert cnn hpc gpu nlp llm model useful


In [16]:
# Process the research-paper dataset while preserving every required column
DATA_PATH = PROJECT_ROOT / "data" / "processed_data" / "merged_research_papers.csv"
papers = pd.read_csv(DATA_PATH)
validate_required_columns(papers)

processed_papers = preprocess_dataframe(papers, lemmatize=False)

assert all(column in processed_papers.columns for column in REQUIRED_COLUMNS)
assert all(column in processed_papers.columns for column in ["title", "abstract", "clean_text", "category", "year"])
assert len(processed_papers) == len(papers)

print("papers =", len(processed_papers))
print("all required columns present =", True)
processed_papers[["Title", "Abstract", "clean_text", "category", "year"]].head()


papers = 46344
all required columns present = True


,Title,Abstract,clean_text,category,year
0,The role of quantum computing in advancing sci...,NaN,role quantum computing advancing scientific hi...,Quantum Computing Algorithms and Architecture,<NA>
1,Queue wait time prediction in high performance...,Abstract High Performance Computing (HPC) syst...,queue wait time prediction high performance co...,Cloud Computing and Resource Management,<NA>
2,A High-Performance Computing Portal Applied to...,Abstract The continued expansion in size and r...,high-performance computing portal applied 3d e...,Advanced Electron Microscopy Techniques and Ap...,<NA>
3,End-edge-cloud collaborative-driven waste-heat...,NaN,end-edge-cloud collaborative-driven waste-heat...,Cloud Computing and Resource Management,<NA>
4,MorphoCloud: Democratizing Access to High-Perf...,Background: The digitization of biological spe...,morphocloud democratizing access high-performa...,"Genetics, Bioinformatics, and Biomedical Research",<NA>


In [ ]:
# Generate the requested local preprocessed dataset
preprocessed_dataset = build_preprocessed_dataset(papers, lemmatize=False)
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
preprocessed_dataset.to_csv(OUTPUT_PATH, index=False)

saved_dataset = pd.read_csv(OUTPUT_PATH)
assert list(saved_dataset.columns) == OUTPUT_COLUMNS
assert len(saved_dataset) == len(papers)
assert set(saved_dataset["Top 1% cited"].dropna().unique()).issubset({0, 1})
assert set(saved_dataset["Top 10% cited"].dropna().unique()).issubset({0, 1})

print("saved to =", OUTPUT_PATH)
print("rows =", len(saved_dataset))
print("columns =", list(saved_dataset.columns))
print("original top 1% values =", papers["Top 1% cited"].dropna().unique()[:10])
print("original top 10% values =", papers["Top 10% cited"].dropna().unique()[:10])
print("output top 1% values =", sorted(saved_dataset["Top 1% cited"].unique()))
print("output top 10% values =", sorted(saved_dataset["Top 10% cited"].unique()))
saved_dataset.head()


saved to = D:\4-1\NLP_Lab\NLP-Based-Research-Paper-Intelligence-System\data\processed_data\preprocessed_research_papers.csv
rows = 46344
columns = ['Title', 'Author', 'Institution', 'Topic', 'Domain', 'Field', 'Concept', 'Abstract', 'Keyword', 'Citation count', 'Cites', 'Top 1% cited', 'Top 10% cited', 'DOI']
top 1% values = [np.int64(0)]
top 10% values = [np.int64(0)]


,Title,Author,Institution,Topic,Domain,Field,Concept,Abstract,Keyword,Citation count,Cites,Top 1% cited,Top 10% cited,DOI
0,role quantum computing advancing scientific hi...,Gilles Buchs|Thomas L. Beck|Ryan S. Bennink|Da...,Oak Ridge National Laboratory|ETH Zurich|CSCS ...,quantum computing algorithms architecture,physical sciences,computer science,perspective graphical computer science quantum...,NaN,perspective graphical quantum computer history...,4,190,0,0,https://doi.org/10.1016/j.future.2026.108487
1,queue wait time prediction high performance co...,Nwamaka U. Okafor|Bethany Lusch|Venkatram Vish...,Argonne National Laboratory,cloud computing resource management,physical sciences,computer science,job queue computer science job scheduler super...,abstract high performance computing hpc system...,job queue job scheduler supercomputer queue sc...,2,13,0,0,https://doi.org/10.1007/s11227-025-08221-7
2,high-performance computing portal applied 3d e...,James P. Carson|Tracy Brown|James A. Labyer|Th...,The University of Texas at Austin|Salk Institu...,advanced electron microscopy techniques applic...,life sciences,biochemistry genetics molecular biology,gateway web page computer science process comp...,abstract continued expansion size resolution v...,gateway web page process computing high resolu...,1,46,0,0,https://doi.org/10.1007/s42979-026-05124-z
3,end-edge-cloud collaborative-driven waste-heat...,Shuaiyin Ma|Mengmeng Zhang|Shi Cheng|Yunran Mi...,Xi’an University of Posts and Telecommunicatio...,cloud computing resource management,physical sciences,computer science,computer science supercomputer data center eff...,NaN,supercomputer data center efficient energy use...,3,59,0,0,https://doi.org/10.1016/j.engappai.2026.114165
4,morphocloud democratizing access high-performa...,A. Murat Maga|Jean‐Christophe Fillion‐Robin,University of Washington|Seattle Children's Ho...,genetics bioinformatics biomedical research,life sciences,biochemistry genetics molecular biology,computer science troubleshooting workflow work...,background digitization biological specimens r...,troubleshooting workflow workstation orchestra...,1,16,0,0,https://doi.org/10.12688/f1000research.176328.1
